# PyTorch Internals: How the Dispatcher Routes Every Operation

**Module 35** | Time: ~3 hours | Prerequisites: Modules 02, 04, 08, 19

---

The dispatcher is the central nervous system of PyTorch. Every call to `torch.add(x, y)` flows through it. This notebook explores how it works, how to inspect it, and how to extend it with custom ops.

```
torch.add(x, y)
    → Dispatcher examines tensor keys
    → Walks priority chain (Autograd → Backend)
    → Executes matching kernel
    → Returns result
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. What is the Dispatcher?

The dispatcher is a **routing table** that maps (operator, dispatch keys) → kernel implementation.

```
┌──────────────────────────────────────────┐
│          User: torch.add(x, y)           │
└─────────────────────┬────────────────────┘
                      │
                      ▼
┌──────────────────────────────────────────┐
│              Dispatcher                   │
│  Keys from x: {CPU, AutogradCPU}        │
│  Keys from y: {CPU, AutogradCPU}        │
│  Union: {CPU, AutogradCPU}              │
│                                          │
│  Walk priority: AutogradCPU → CPU       │
└─────────────────────┬────────────────────┘
                      │
          ┌───────────┴───────────┐
          ▼                       ▼
   AutogradCPU kernel       CPU kernel
   (record for backward)    (compute result)
```

## 2. Dispatch Keys on Different Tensors

Each tensor carries a **dispatch key set** — a bitset of active features.

In [ ]:
# Basic CPU tensor
x = torch.randn(3, 3)
print(f"CPU tensor (no grad): {torch._C._dispatch_keys(x)}")

# With requires_grad
x_grad = torch.randn(3, 3, requires_grad=True)
print(f"CPU tensor (grad):    {torch._C._dispatch_keys(x_grad)}")

# Meta tensor
x_meta = torch.randn(3, 3, device='meta')
print(f"Meta tensor:          {torch._C._dispatch_keys(x_meta)}")

# Integer tensor (no autograd)
x_int = torch.randint(0, 10, (3, 3))
print(f"Int tensor:           {torch._C._dispatch_keys(x_int)}")

In [ ]:
# Adding requires_grad changes the key set
t = torch.randn(4, 4)
print(f"Before requires_grad_(): {torch._C._dispatch_keys(t)}")
t.requires_grad_(True)
print(f"After requires_grad_():  {torch._C._dispatch_keys(t)}")

In [ ]:
# CUDA tensor (if available)
if torch.cuda.is_available():
    x_cuda = torch.randn(3, 3, device='cuda')
    print(f"CUDA tensor:         {torch._C._dispatch_keys(x_cuda)}")
    x_cuda_g = torch.randn(3, 3, device='cuda', requires_grad=True)
    print(f"CUDA tensor (grad):  {torch._C._dispatch_keys(x_cuda_g)}")
else:
    print("CUDA not available")

## 3. The Priority Chain

Keys are walked from highest to lowest priority. First key with a registered kernel wins.

| Priority | Key | Purpose |
|----------|-----|--------|
| Highest | PythonTLSSnapshot | Thread-local state |
| High | PythonDispatcher | torch.compile |
| | Autocast | Mixed precision |
| | AutogradCPU/CUDA | Record for backward |
| | ADInplaceOrView | Track views/in-place |
| | BackendSelect | Route factory ops |
| Low | CPU/CUDA/MPS/Meta | Actual compute |
| Lowest | CompositeImplicit | Decompositions |

In [ ]:
# Verify kernel registrations for aten::add
op = "aten::add.Tensor"
for key in ["AutogradCPU", "AutogradCUDA", "CPU", "CUDA", "Meta",
            "CompositeImplicitAutograd", "BackendSelect"]:
    try:
        has = torch._C._dispatch_has_kernel_for_dispatch_key(op, key)
        print(f"  {key:35s} → {'REGISTERED' if has else 'fallthrough'}")
    except Exception:
        print(f"  {key:35s} → (unavailable)")

## 4. Dump Dispatch Tables

Inspect what kernels are registered for any operator.

In [ ]:
# Full dispatch table for aten::add.Tensor
print("=== aten::add.Tensor ===")
table = torch._C._dispatch_dump("aten::add.Tensor")
for line in table.strip().split('\n')[:20]:
    print(f"  {line}")

In [ ]:
# Compare with torch.mm
print("=== aten::mm ===")
table = torch._C._dispatch_dump("aten::mm")
for line in table.strip().split('\n')[:20]:
    print(f"  {line}")

In [ ]:
# Check relu
print("=== aten::relu ===")
table = torch._C._dispatch_dump("aten::relu")
for line in table.strip().split('\n')[:20]:
    print(f"  {line}")

## 5. Custom Op with torch.library

The `Library` API lets you register ops in the dispatcher with full backend support.

In [ ]:
from torch.library import Library, impl

# Define op in custom namespace
lib = Library("notebook35", "DEF")
lib.define("scaled_add(Tensor x, Tensor y, float scale) -> Tensor")

# CPU implementation
@impl(lib, "scaled_add", "CPU")
def scaled_add_cpu(x, y, scale):
    return x + y * scale

# Meta implementation
@impl(lib, "scaled_add", "Meta")
def scaled_add_meta(x, y, scale):
    return torch.empty_like(x)

# Use it
x = torch.randn(4, 4)
y = torch.randn(4, 4)
result = torch.ops.notebook35.scaled_add(x, y, scale=2.0)
print(f"Result matches x + 2*y: {torch.allclose(result, x + 2*y)}")
print(f"Result shape: {result.shape}")

In [ ]:
# View the dispatch table for our custom op
try:
    table = torch._C._dispatch_dump("notebook35::scaled_add")
    for line in table.strip().split('\n')[:15]:
        print(line)
except Exception as e:
    print(f"Could not dump: {e}")

## 6. @custom_op — The Modern API

PyTorch 2.4+ provides a simpler decorator-based registration.

In [ ]:
@torch.library.custom_op("notebook35_modern::fast_gelu", mutates_args=())
def fast_gelu(x: torch.Tensor) -> torch.Tensor:
    return x * torch.sigmoid(1.702 * x)

@fast_gelu.register_fake
def fast_gelu_fake(x):
    return torch.empty_like(x)

# Test it
x = torch.randn(8, 8)
result = torch.ops.notebook35_modern.fast_gelu(x)
expected = x * torch.sigmoid(1.702 * x)
print(f"Correct: {torch.allclose(result, expected)}")
print(f"Shape: {result.shape}")

## 7. register_fake for Meta/Shape Inference

The "fake" (Meta) implementation tells torch.compile what output shape/dtype to expect.

In [ ]:
# Meta tensors use the fake implementation
x_meta = torch.randn(16, 32, device='meta')
result_meta = torch.ops.notebook35_modern.fast_gelu(x_meta)
print(f"Input:  shape={x_meta.shape}, device={x_meta.device}")
print(f"Output: shape={result_meta.shape}, device={result_meta.device}")
print(f"\nMeta dispatch computes shapes without actual data!")
print(f"This is exactly what torch.compile uses during tracing.")

## 8. Autograd Registration

Register backward pass for custom ops so they work with `requires_grad=True`.

In [ ]:
# Register autograd for fast_gelu
def fast_gelu_setup_context(ctx, inputs, output):
    x, = inputs
    ctx.save_for_backward(x)

def fast_gelu_backward(ctx, grad_output):
    x, = ctx.saved_tensors
    sig = torch.sigmoid(1.702 * x)
    grad = sig + 1.702 * x * sig * (1 - sig)
    return (grad_output * grad,)

fast_gelu.register_autograd(
    fast_gelu_backward,
    setup_context=fast_gelu_setup_context,
)

# Test autograd
x = torch.randn(4, 4, requires_grad=True)
y = torch.ops.notebook35_modern.fast_gelu(x)
y.sum().backward()
print(f"x.grad shape: {x.grad.shape}")
print(f"x.grad has values: {x.grad.abs().mean().item():.4f} (mean abs)")

# Verify against numerical gradient
x2 = x.detach().clone().requires_grad_(True)
y2 = x2 * torch.sigmoid(1.702 * x2)
y2.sum().backward()
print(f"Matches manual grad: {torch.allclose(x.grad, x2.grad, atol=1e-5)}")

## 9. How torch.compile Bypasses Dispatch

torch.compile captures a graph of operations and generates code that calls kernels directly — bypassing the dispatcher at runtime.

In [ ]:
class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(16, 16)

    def forward(self, x):
        x = self.linear(x)
        x = torch.ops.notebook35_modern.fast_gelu(x)
        return x

model = TinyModel()
x = torch.randn(4, 16)

# Eager mode
out_eager = model(x)

# Compiled mode (uses register_fake for tracing)
compiled = torch.compile(model, backend="eager")
out_compiled = compiled(x)

print(f"Eager output:    {out_eager.shape}")
print(f"Compiled output: {out_compiled.shape}")
print(f"Match: {torch.allclose(out_eager, out_compiled, atol=1e-6)}")
print(f"\ntorch.compile used register_fake during tracing,")
print(f"then calls the real kernel at runtime.")

## 10. Dispatch Trace with TorchDispatchMode

`TorchDispatchMode` intercepts ops at the dispatch layer — below Python, below autograd.

In [ ]:
from torch.utils._python_dispatch import TorchDispatchMode

class OpLogger(TorchDispatchMode):
    def __init__(self):
        self.ops = []

    def __torch_dispatch__(self, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}
        self.ops.append(func.name())
        return func(*args, **kwargs)

# Trace a model forward pass
model = nn.Sequential(
    nn.Linear(16, 32),
    nn.ReLU(),
    nn.Linear(32, 8),
)
x = torch.randn(4, 16)

logger = OpLogger()
with logger:
    out = model(x)

print(f"Forward pass dispatched {len(logger.ops)} ops:")
for i, op in enumerate(logger.ops, 1):
    print(f"  {i}. {op}")

In [ ]:
# Compare: TorchFunctionMode sees higher-level Python calls
from torch.overrides import TorchFunctionMode

class FuncLogger(TorchFunctionMode):
    def __init__(self):
        self.ops = []

    def __torch_function__(self, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}
        name = func.__name__ if hasattr(func, '__name__') else str(func)
        self.ops.append(name)
        return func(*args, **kwargs)

func_logger = FuncLogger()
x = torch.randn(4, 16)
with func_logger:
    out = model(x)

print(f"TorchFunctionMode saw {len(func_logger.ops)} ops:")
for i, op in enumerate(func_logger.ops[:10], 1):
    print(f"  {i}. {op}")
if len(func_logger.ops) > 10:
    print(f"  ... and {len(func_logger.ops) - 10} more")

## 11. Building a Simple Op Profiler

In [ ]:
import time

class SimpleProfiler(TorchDispatchMode):
    def __init__(self):
        self.timings = {}

    def __torch_dispatch__(self, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}
        name = func.name()
        start = time.perf_counter_ns()
        result = func(*args, **kwargs)
        elapsed = (time.perf_counter_ns() - start) / 1000  # microseconds
        if name not in self.timings:
            self.timings[name] = {'count': 0, 'total_us': 0.0}
        self.timings[name]['count'] += 1
        self.timings[name]['total_us'] += elapsed
        return result

# Profile a larger model
model = nn.Sequential(
    nn.Linear(64, 128), nn.GELU(),
    nn.Linear(128, 128), nn.LayerNorm(128),
    nn.Linear(128, 32),
)

profiler = SimpleProfiler()
x = torch.randn(16, 64)
with profiler:
    for _ in range(20):
        out = model(x)

# Top ops by time
sorted_ops = sorted(profiler.timings.items(), key=lambda x: -x[1]['total_us'])
print(f"{'Op':<40s} {'Calls':>6s} {'Total (us)':>10s}")
print("-" * 60)
for name, stats in sorted_ops[:10]:
    print(f"{name:<40s} {stats['count']:>6d} {stats['total_us']:>10.0f}")

## 12. CompositeImplicit vs Backend-Specific Kernels

In [ ]:
# Some ops have backend-specific kernels (optimized)
# Others use CompositeImplicitAutograd (decompose into primitives)

ops_to_check = [
    ("aten::add.Tensor", "Has CPU+CUDA kernels"),
    ("aten::mm", "Has CPU+CUDA kernels"),
    ("aten::selu", "Likely CompositeImplicit"),
    ("aten::mish", "Likely CompositeImplicit"),
]

for op_name, desc in ops_to_check:
    print(f"\n{op_name} ({desc}):")
    for key in ["CPU", "CUDA", "CompositeImplicitAutograd"]:
        try:
            has = torch._C._dispatch_has_kernel_for_dispatch_key(op_name, key)
            print(f"  {key:35s} → {'YES' if has else 'no'}")
        except Exception:
            print(f"  {key:35s} → (unavailable)")

## 13. Meta Tensors — Shape Dispatch Without Compute

In [ ]:
# Meta tensors only track shape/dtype — no actual data
a = torch.randn(3, 4, device='meta')
b = torch.randn(4, 5, device='meta')

# Operations compute output shapes
c = torch.mm(a, b)
print(f"mm({a.shape}, {b.shape}) → {c.shape}")

d = torch.cat([a, torch.randn(2, 4, device='meta')], dim=0)
print(f"cat([{a.shape}, (2,4)], dim=0) → {d.shape}")

e = a.unsqueeze(0).expand(8, -1, -1)
print(f"unsqueeze+expand → {e.shape}")

print(f"\nThis is how torch.compile traces shapes through your model!")

## 14. Autograd and Dispatch Interaction

In [ ]:
x = torch.randn(3, 3, requires_grad=True)
y = torch.randn(3, 3, requires_grad=True)

# The AutogradCPU dispatch key records ops for backward
z = x @ y + x
print(f"z.grad_fn: {z.grad_fn}")
print(f"z.grad_fn.next_functions: {z.grad_fn.next_functions}")
print()

# torch.no_grad() excludes Autograd keys from dispatch
with torch.no_grad():
    w = x @ y + x
    print(f"With no_grad - w.grad_fn: {w.grad_fn}")
    print(f"  → AutogradCPU key excluded from dispatch!")

## 15. Exercise: Register a Fused Add-ReLU Op

**Task**: Register `fused_add_relu(x, y) = relu(x + y)` with:
1. CPU kernel
2. Meta (fake) kernel
3. Autograd support

Then verify it works with `torch.compile`.

In [ ]:
# Exercise: Complete this custom op registration

@torch.library.custom_op("exercise35::fused_add_relu", mutates_args=())
def fused_add_relu(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return torch.relu(x + y)

# TODO: Register fake implementation
@fused_add_relu.register_fake
def fused_add_relu_fake(x, y):
    return torch.empty_like(x)

# TODO: Register autograd
def fused_add_relu_setup_context(ctx, inputs, output):
    x, y = inputs
    ctx.save_for_backward(output)

def fused_add_relu_backward(ctx, grad_output):
    output, = ctx.saved_tensors
    grad = grad_output * (output > 0).float()
    return grad, grad

fused_add_relu.register_autograd(
    fused_add_relu_backward,
    setup_context=fused_add_relu_setup_context,
)

# Verify
x = torch.randn(4, 4, requires_grad=True)
y = torch.randn(4, 4, requires_grad=True)
out = torch.ops.exercise35.fused_add_relu(x, y)
out.sum().backward()

print(f"Output correct: {torch.allclose(out, torch.relu(x.detach() + y.detach()))}")
print(f"Grad correct: {torch.allclose(x.grad, ((x.detach() + y.detach()) > 0).float())}")

# torch.compile test
@torch.compile(backend="eager")
def compiled_fused(a, b):
    return torch.ops.exercise35.fused_add_relu(a, b)

a, b = torch.randn(4, 4), torch.randn(4, 4)
print(f"Compile works: {torch.allclose(compiled_fused(a, b), torch.relu(a + b))}")

## 16. Inspecting Dispatch for Debugging

In [ ]:
# Common debugging patterns

# 1. Check what dispatch keys a tensor has
t = torch.randn(3, 3, requires_grad=True)
print(f"Keys: {torch._C._dispatch_keys(t)}")

# 2. Check if an op has a kernel for a specific key
print(f"\naten::add.Tensor has CPU kernel: "
      f"{torch._C._dispatch_has_kernel_for_dispatch_key('aten::add.Tensor', 'CPU')}")

# 3. List available ops in a namespace
aten_ops = [op for op in dir(torch.ops.aten) if not op.startswith('_')]
print(f"\naten namespace has {len(aten_ops)} ops")
print(f"First 10: {aten_ops[:10]}")

## Key Takeaways

1. **Every PyTorch op goes through the dispatcher** — it's the routing layer
2. **Dispatch keys** are a bitset on each tensor (device + features like autograd)
3. **Priority chain** — highest key with a kernel wins, then can redispatch
4. **torch.library** and `@custom_op` — register your own ops in the dispatcher
5. **register_fake** — required for torch.compile (shape inference via Meta tensors)
6. **TorchDispatchMode** — intercept all ops at the dispatch level for debugging
7. **torch.compile bypasses most dispatch** at runtime (generates direct kernel calls)
8. **The dispatcher is what makes PyTorch extensible** — backends, autograd, compile all plug in

---

**Next Steps**: Experiment with registering custom ops for your models, use `TorchDispatchMode` for profiling, and explore how `torch.compile` interacts with your custom ops.